# Example queries: `helpers` (comstock_oedi)

Auto-generated from `tests/query_snapshots/helpers.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [1]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [2]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/comstock_oedi_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/comstock_oedi_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "comstock_amy2018_r2_2025",
    buildstock_type="comstock",
    db_schema="comstock_oedi_state_and_county",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


INFO:buildstock_query.query_core:Loading comstock_amy2018_r2_2025 ...


INFO:botocore.tokens:Loading cached SSO token for nrel-sso


## `upgrade_names_oedi`

get_upgrade_names — was broken before 2026-04-26 fix on two counts: (1) f-string interpolation of self.up_table (a Subquery) produced malformed `FROM SELECT * FROM ...` SQL with no enclosing parens; (2) the method assumed an `apply_upgrade.upgrade_name` column that doesn't exist on OEDI schemas. Fixed to use SA construction and gracefully degrade to `CAST(NULL AS VARCHAR) AS upgrade_name` when the name column is absent — so the dict shape stays consistent across schemas.


In [3]:
result = bsq.get_upgrade_names()
result.head() if hasattr(result, 'head') else result


{1: 'Variable Speed HP RTU, Electric Backup',
 2: 'Variable Speed HP RTU, Original Heating Fuel Backup',
 3: 'Variable Speed HP RTU, Electric Backup, Energy Recovery',
 4: 'Standard Performance HP RTU, Electric Backup',
 5: 'Standard Performance HP RTU using Lab Data, Electric Backup',
 6: 'Standard Performance HP RTU, Electric Backup + Roof Insulation',
 7: 'Standard Performance HP RTU, Electric Backup + New Windows',
 8: 'Standard Performance HP RTU, Electric Backup, 32F Minimum Compressor Lockout',
 9: 'Standard Performance HP RTU, Electric Backup, 2F Unoccupied Htg Thermostat Setback',
 10: 'Cold Climate Challenge HP RTU, Electric Backup',
 11: 'VRF with DOAS',
 12: 'VRF with 25pct Upsizing Allowance',
 13: 'DOAS HP Minisplits',
 14: 'HP Boiler, Electric Backup',
 15: 'HP Boiler, Gas Backup',
 16: 'Condensing Gas Boilers',
 17: 'Electric Resistance Boilers',
 18: 'Air Side Economizers for AHUs',
 19: 'Demand Control Ventilation',
 20: 'Energy Recovery for AHUs',
 21: 'Advanced RTU 

## `upgrades_csv_three_buildings_upgrade1`

get_upgrades_csv for upgrade=1 restricted to a tiny known building_id list. Mirrors results_csv_three_buildings on the upgrade-table side.


In [4]:
result = bsq.get_upgrades_csv(upgrade_id='1', restrict=[('bldg_id', [1, 2, 3])])
result.head() if hasattr(result, 'head') else result


INFO:buildstock_query.main:Making results_csv query for upgrade ...


KeyError: "None of ['bldg_id', 'in.nhgis_tract_gisjoin', 'state'] are in the columns"

## `buildings_by_locations_state`

get_buildings_by_locations: list bs-keys whose location_col=$STATE_COL_BASELINE is in ['CO']. Different code path from restrict — uses bs_key_cols projection and IN-list filter directly.


In [5]:
result = bsq.get_buildings_by_locations(location_col='state', locations=['CO'])
result.head() if hasattr(result, 'head') else result


,bldg_id,in.nhgis_tract_gisjoin,state
0,50216,G0800410007201,CO
1,50216,G0800410007700,CO
2,50216,G0800410007800,CO
3,50216,G0800590010206,CO
4,50216,G0800590010304,CO


## `results_csv_three_buildings`

get_results_csv restricted to a tiny known building_id list. Pins the SELECT-* shape of the baseline metadata projection used by basic_usage.ipynb. Bounded to 3 buildings so the data parquet stays small (results_csv has ~150 columns, full state would be 100+ MB).


In [6]:
result = bsq.get_results_csv(restrict=[('bldg_id', [1, 2, 3])])
result.head() if hasattr(result, 'head') else result


INFO:buildstock_query.main:Making results_csv query ...


KeyError: "None of ['bldg_id', 'in.nhgis_tract_gisjoin', 'state'] are in the columns"

## `distinct_vals_state_baseline`

get_distinct_vals on the baseline-table state column. Pins the simple SELECT DISTINCT shape — used by notebooks to enumerate categorical values before building restricts. The column name differs by schema (resstock=`in.state`, comstock=`state`); resolved via $STATE_COL_BASELINE.


In [7]:
result = bsq.get_distinct_vals(
    column='state',
    table_name='comstock_amy2018_r2_2025_md_by_state_and_county_parquet',
)
result.head() if hasattr(result, 'head') else result


0    RI
1    ME
2    AK
3    AR
4    HI
Name: state, dtype: str

## `distinct_count_state_baseline`

get_distinct_count on the baseline-table state column. Pins the COUNT-with-weighted-sum shape (sample_count + weighted_count per category).


In [8]:
result = bsq.get_distinct_count(column='state')
result.head() if hasattr(result, 'head') else result


,state,sample_count,weighted_count
0,AK,13248,8113.081188
1,AL,163198,98577.243244
2,AR,92920,58124.142056
3,AZ,152972,86181.239323
4,CA,746595,449744.576018
